In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

def run_command(args):
    try:
        completed = subprocess.run(args, capture_output=True, text=True, timeout=30)
        return {
            "ok": completed.returncode == 0,
            "returncode": completed.returncode,
            "stdout": completed.stdout.strip(),
            "stderr": completed.stderr.strip(),
        }
    except Exception as exc:
        return {"ok": False, "error": repr(exc)}

report = {
    "status": "started",
    "utc_time": datetime.now(timezone.utc).isoformat(),
    "cwd": os.getcwd(),
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "env": {
        "KAGGLE_KERNEL_RUN_TYPE": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
        "KAGGLE_URL_BASE": os.environ.get("KAGGLE_URL_BASE"),
        "CUDA_VISIBLE_DEVICES": os.environ.get("CUDA_VISIBLE_DEVICES"),
    },
    "nvidia_smi": run_command(["nvidia-smi"]),
}

try:
    import torch
    cuda_available = torch.cuda.is_available()
    devices = []
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        devices.append({
            "index": index,
            "name": torch.cuda.get_device_name(index),
            "total_memory_bytes": props.total_memory,
            "major": props.major,
            "minor": props.minor,
            "multi_processor_count": props.multi_processor_count,
        })
    report["torch"] = {
        "imported": True,
        "version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "cuda_available": cuda_available,
        "device_count": torch.cuda.device_count(),
        "devices": devices,
    }
except Exception as exc:
    report["torch"] = {"imported": False, "error": repr(exc)}

report["status"] = "ok" if report.get("torch", {}).get("cuda_available") else "no_cuda"
Path("gpu_probe_result.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
